# Complementaria Semana 4: Introducción a SimPy

En la magistral de la semana 3 vimos los conceptos de la **simulación de eventos discretos (SED)**: eventos, reloj de simulación, lista de eventos futuros (FEL) y el algoritmo de avance del tiempo. Hoy vamos a ver cómo se traduce todo eso a código con **SimPy**.

### Objetivo de hoy

Al final de la sesión deberías poder leer un modelo de SimPy y saber **dónde está el reloj, dónde están los procesos y dónde avanza el tiempo**.

| Bloque | Tema |
|---|---|
| 1 | El entorno (`Environment`) y tu primer proceso |
| 2 | Varios procesos a la vez: el reloj *salta* entre eventos |
| 3 | Llegadas aleatorias: interarribos exponenciales |
| 4 | Recursos compartidos: `Resource` y la cola |
| 5 | Caso Cafetería: `Container` y el cuello de botella |
| 6 | Ejercicio guiado |

## 0. Preparación

Este notebook se trabaja en **VS Code**, sobre el **ambiente virtual** que creaste en los tutoriales de la semana 1. La única librería nueva de hoy es SimPy.

Corre la celda de abajo una sola vez. Usamos `%pip` (con `%`, no con `!`) porque así la instalación va al ambiente virtual que está usando el kernel de este notebook, y no al Python global del computador.

> Si te sale un error de importación al correr la celda siguiente, puedes hacer click en `Restart` kernel y volver a correr la celda de importación.

In [ ]:
%pip install simpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
import simpy
import random
import statistics
import pandas as pd

# Función auxiliar: imprime un mensaje junto con el tiempo simulado actual.
# La vamos a usar todo el tiempo para "ver" cómo avanza el reloj.
def log(env, mensaje):
    print(f"t = {env.now:6.2f} | {mensaje}")

## 1. El entorno y tu primer proceso

En SimPy hay dos piezas fundamentales:

- **`Environment` (entorno)**: es el reloj de la simulación y el planificador de eventos. Es quien decide qué pasa y cuándo. Es la implementación concreta de la FEL que vieron en la magistral.
- **Proceso**: es el comportamiento de una entidad del sistema (un cliente, una máquina, un barista). En SimPy un proceso se escribe como una **función generadora**: una función que contiene `yield`.

Cada `yield` le dice al entorno: *"pausa este proceso y reanúdalo cuando ocurra este evento"*. El evento más simple es `env.timeout(t)`, que significa **"espera `t` unidades de tiempo simulado"**.

Veamos el ejemplo más pequeño posible. Fíjate en la salida: la línea de "EMPIEZA" y la de "TERMINA" están separadas por 5 unidades de tiempo, aunque el computador las ejecuta en microsegundos. **El tiempo simulado no tiene nada que ver con el tiempo real.**

In [9]:
# Un proceso: el barista prepara un café que tarda 5 minutos
def preparar_cafe(env):
    log(env, "El barista EMPIEZA a preparar el café")
    yield env.timeout(5)                  # <- aquí avanza el reloj
    log(env, "El barista TERMINA el café")

# 1) Creamos el entorno (el reloj + el planificador)
env = simpy.Environment()

# 2) Registramos el proceso en el entorno
env.process(preparar_cafe(env))

# 3) Corremos la simulación
log(env, "--- arranca la simulación ---")
env.run(until=100)
log(env, "--- fin de la simulación ---")

t =   0.00 | --- arranca la simulación ---
t =   0.00 | El barista EMPIEZA a preparar el café
t =   5.00 | El barista TERMINA el café
t = 100.00 | --- fin de la simulación ---


### ¿Qué hizo cada línea?

| Línea | Qué hace |
|---|---|
| `def preparar_cafe(env):` | Define el **comportamiento** de una entidad. Todavía no ocurre nada. |
| `yield env.timeout(5)` | Crea un evento "dentro de 5 unidades de tiempo" y **cede el control** al entorno. |
| `env = simpy.Environment()` | Crea el reloj. Arranca en `env.now = 0`. |
| `env.process(preparar_cafe(env))` | Registra el proceso para que el entorno lo ejecute. |
| `env.run(until=100)` | Ejecuta la simulación hasta $t = 100$. |

**Ojo con `until`.** El reloj terminó en $t = 100$, **no** en $t = 5$. Cuando le pasas `until=100`, SimPy agenda internamente un evento de parada en $t=100$ y corre hasta allá, así el sistema haya quedado inactivo desde $t=5$. Si en cambio llamas `env.run()` **sin argumentos**, la simulación corre hasta que se acabe la lista de eventos y el reloj queda en el último evento. Compruébalo:

In [10]:
env = simpy.Environment()
env.process(preparar_cafe(env))
env.run()                                  # <- sin until
print("Reloj final sin 'until':", env.now)

t =   0.00 | El barista EMPIEZA a preparar el café
t =   5.00 | El barista TERMINA el café
Reloj final sin 'until': 5


> **Ejercicio rápido.** Cambia el `5` del `timeout` por `12` y vuelve a correr ambas celdas. ¿En qué instante aparece el mensaje de "TERMINA"? ¿Cambia el reloj final de la primera celda? ¿Y de la segunda?

## 2. Varios procesos a la vez: el reloj *salta*

Aquí está la idea central de la SED. Registramos **dos máquinas** trabajando en paralelo con ciclos distintos: A produce una pieza cada 3 minutos y B cada 5 minutos.

Antes de correr la celda, predice: ¿en qué instantes va a imprimir algo?

In [ ]:
# Definimos un proceso que representa una máquina que produce piezas.
def maquina(env, nombre, ciclo):
    pieza = 0
    while True:                            # el proceso vive indefinidamente
        yield env.timeout(ciclo)           # tarda 'ciclo' minutos en producir
        pieza += 1
        log(env, f"{nombre} termina la pieza #{pieza}")

env = simpy.Environment()                   # creamos el entorno
env.process(maquina(env, "Máquina A", 3))   # registramos el proceso A
env.process(maquina(env, "Máquina B", 5))   # registramos el proceso B
env.run(until=16)                           # corremos la simulación hasta t=16

t =   3.00 | Máquina A termina la pieza #1
t =   5.00 | Máquina B termina la pieza #1
t =   6.00 | Máquina A termina la pieza #2
t =   9.00 | Máquina A termina la pieza #3
t =  10.00 | Máquina B termina la pieza #2
t =  12.00 | Máquina A termina la pieza #4
t =  15.00 | Máquina B termina la pieza #3
t =  15.00 | Máquina A termina la pieza #5


Observa la salida con cuidado:

- El reloj **solo tomó los valores 3, 5, 6, 9, 10, 12, 15**. Nunca valió 3.5, ni 4, ni 7. Esos instantes son exactamente los eventos de la FEL: el simulador **salta** de un evento al siguiente y no pierde tiempo en el medio. Esto es lo que hace que la SED sea eficiente.
- En $t = 15$ ocurren **dos eventos simultáneos** (A termina su pieza 5 y B su pieza 3). SimPy los procesa en el orden en que fueron agendados, pero el reloj no avanza entre ellos.
- Los dos procesos se escribieron con **la misma función** (maquina). Un proceso es una plantilla de comportamiento; puedes usarla tantas veces como quieras.

### Regla de oro:

> Si dentro de un proceso no hay un `yield`, ese código ocurre instantáneamente en tiempo simulado. 

El reloj **solo** avanza cuando un proceso hace `yield` sobre un evento. Todo lo demás (cálculos, `print`, asignaciones) ocurre "congelado" en el instante actual:

In [ ]:
def calcular_sin_esperar(env):
    for i in range(3):
        log(env, f"paso {i}: hago una cuenta, pero el reloj NO avanza")
    yield env.timeout(0)

env = simpy.Environment()
env.process(calcular_sin_esperar(env))       # registramos el proceso 
env.run()

t =   0.00 | paso 0: hago una cuenta, pero el reloj NO avanza
t =   0.00 | paso 1: hago una cuenta, pero el reloj NO avanza
t =   0.00 | paso 2: hago una cuenta, pero el reloj NO avanza


## 3. Llegadas aleatorias: interarribos exponenciales

Hasta ahora los tiempos eran fijos. En un sistema real las llegadas son aleatorias.

De la magistral sabemos que si los clientes llegan según un **proceso de Poisson con tasa $\lambda$**, entonces los **tiempos entre llegadas (interarribos)** son exponenciales con media $1/\lambda$. En Python, `random.expovariate(lam)` genera exactamente eso.

In [ ]:
def generador_llegadas(env, lam, n_max):
    for i in range(1, n_max + 1):
        yield env.timeout(random.expovariate(lam))   # interarribo ~ Exp(lam) -> espera a que llegue el siguiente cliente
        log(env, f"llega el cliente {i}")            # registramos la llegada del cliente

random.seed(4)                                       # semilla: resultados reproducibles
env = simpy.Environment()
env.process(generador_llegadas(env, lam=0.5, n_max=7))  # registramos el proceso de llegadas
env.run()

t =   0.54 | llega el cliente 1
t =   0.76 | llega el cliente 2
t =   1.76 | llega el cliente 3
t =   2.10 | llega el cliente 4
t =   2.24 | llega el cliente 5
t =   3.27 | llega el cliente 6
t =   8.27 | llega el cliente 7


Fíjate en que los interarribos son muy irregulares: entre el cliente 1 y el 2 pasa menos de un minuto, y luego hay un hueco de 5 minutos. Esa es justamente la variabilidad que hace que los sistemas se congestionen y que un análisis con promedios se quede corto.

**Verificación rápida** (usando lo aprendido en semana 3): si generamos muchos interarribos, su promedio debería acercarse a $1/\lambda$.

In [14]:
lam = 0.5
random.seed(0)
interarribos = [random.expovariate(lam) for _ in range(100_000)]

print(f"Media simulada : {statistics.mean(interarribos):.4f}")
print(f"Media teórica  : {1/lam:.4f}")

Media simulada : 1.9980
Media teórica  : 2.0000


## 4. Recursos compartidos: `Resource` y la cola

Nos falta la pieza que convierte esto en un sistema de colas: un **recurso limitado** por el que los clientes compiten.

`simpy.Resource(env, capacity=s)` representa $s$ servidores en paralelo con una **cola FIFO interna que SimPy administra por ti**. El patrón es:

```python
with recurso.request() as turno:    # solicito el recurso (servidor) -> barista
    yield turno          # espero hasta que me toque
    ...                  # aquí ya tengo el recurso: lo uso
# al salir del 'with', el recurso se libera automáticamente
```

Dos detalles importantes:

- `yield turno` es donde el cliente **espera en la cola**. Si el servidor está libre, pasa de inmediato (espera cero).
- Usar `with` garantiza que el recurso se libere aunque el proceso termine de forma inesperada. Si no usas `with`, tienes que llamar `recurso.release(turno)` a mano y es fácil olvidarlo.

En el siguiente ejemplo imprimimos la traza completa para **ver la cola formarse y vaciarse**:

In [ ]:
# Creamos un proceso que representa a un cliente que llega a la cafetería, espera a ser atendido por el barista, es atendido y se va.
def cliente(env, nombre, barista):
    log(env, f"{nombre} llega y se pone en la cola")

    with barista.request() as turno:                    # <- solicita el recurso (el barista)
        yield turno                                     # <- espera en la cola
        log(env, f"{nombre} EMPIEZA a ser atendido (quedan {len(barista.queue)} esperando)")

        yield env.timeout(random.expovariate(1/2.0))    # servicio ~ Exp(media 2 min) -> espera a que termine el servicio
        log(env, f"{nombre} se va")

# Creamos un proceso que genera clientes y los envía a la cola del barista
def llegadas(env, barista, lam, n_max):
    for i in range(1, n_max + 1):
        yield env.timeout(random.expovariate(lam))
        env.process(cliente(env, f"Cliente {i}", barista))  # registramos el proceso del cliente que llega


# Simulamos la cafetería con un solo barista y llegadas ~ Exp(0.8) hasta 6 clientes
random.seed(7)
env = simpy.Environment()
barista = simpy.Resource(env, capacity=1)               # un solo servidor
env.process(llegadas(env, barista, lam=0.8, n_max=6))   # registramos el proceso de llegadas (que a su vez registra los procesos de los clientes atendidos)
env.run()

t =   0.49 | Cliente 1 llega y se pone en la cola
t =   0.49 | Cliente 1 EMPIEZA a ser atendido (quedan 0 esperando)
t =   0.69 | Cliente 2 llega y se pone en la cola
t =   0.79 | Cliente 3 llega y se pone en la cola
t =   1.75 | Cliente 4 llega y se pone en la cola
t =   2.32 | Cliente 5 llega y se pone en la cola
t =   2.39 | Cliente 6 llega y se pone en la cola
t =   2.59 | Cliente 1 se va
t =   2.59 | Cliente 2 EMPIEZA a ser atendido (quedan 4 esperando)
t =   4.01 | Cliente 2 se va
t =   4.01 | Cliente 3 EMPIEZA a ser atendido (quedan 3 esperando)
t =   4.09 | Cliente 3 se va
t =   4.09 | Cliente 4 EMPIEZA a ser atendido (quedan 2 esperando)
t =   5.22 | Cliente 4 se va
t =   5.22 | Cliente 5 EMPIEZA a ser atendido (quedan 1 esperando)
t =   5.37 | Cliente 5 se va
t =   5.37 | Cliente 6 EMPIEZA a ser atendido (quedan 0 esperando)
t =   5.56 | Cliente 6 se va


Lee la traza de arriba de corrido:

1. El cliente 1 llega y entra **de inmediato** (espera cero).
2. Los clientes 2 a 6 llegan mientras 1 está siendo atendido y **se acumulan**: la cola llega a tener 5 personas.
3. Cuidado con el número que imprime la traza: cuando el cliente 2 empieza a ser atendido dice `quedan 4 esperando`, no 5. No es un error. `len(barista.queue)` se evalúa **después** de que SimPy ya le entregó el barista al cliente 2, así que él ya salió de la cola y solo quedan los clientes 3 a 6. **El instante en que mides cambia lo que mides** — es un detalle menor aquí, pero la próxima semana, cuando calculemos $L_q$ en serio, va a ser justo lo que hay que cuidar.
4. Cada vez que alguien se va, el siguiente entra **en el mismo instante** (`t` no cambia entre "se va" y "EMPIEZA"). El servidor no descansa mientras haya cola.
5. Se atienden en el orden en que llegaron: FIFO.

Con estas cuatro piezas — `Environment`, procesos, `timeout` y `Resource` — ya puedes modelar una cola M/M/1 completa. **Eso es exactamente lo que haremos la próxima semana**, agregándole la medición de $W_q$, $W$, $L_q$, $L$ y la comparación contra los resultados teóricos.

## 5. Caso Cafetería

### Enunciado

Una cafetería del campus tiene **un barista**. Los clientes llegan según un proceso de Poisson y **todas** las bebidas de la carta llevan leche. La leche se guarda en una nevera que un proveedor repone en **lotes**, cada cierto tiempo fijo.

Si al momento de preparar la bebida no hay leche suficiente, el barista **no puede seguir**: se queda esperando el próximo lote, y como el cliente que está atendiendo no se va, la cola detrás se queda quieta también.

| Parámetro | Símbolo | Valor base | Unidad |
|---|---|---|---|
| Tasa de llegadas | `lam` | 0.9 | clientes / min |
| Tasa de servicio del barista | `mu` | 2.0 | bebidas / min |
| Baristas | `s` | 1 | servidores |
| Leche por bebida | `consumo` | 200 | ml |
| Tamaño del lote | `lote` | 1500 | ml |
| Tiempo entre reposiciones | `R` | 8 | min |
| Capacidad de la nevera | `nevera` | 3000 | ml |

### Lo nuevo: `Container`

Un `Resource` sirve para cosas que se **prestan y se devuelven** (un barista, una caja, una máquina). Para cosas que se **consumen y se reponen** (leche, combustible, inventario) SimPy tiene `Container`:

- `simpy.Container(env, init=..., capacity=...)` — nivel inicial y capacidad máxima.
- `yield contenedor.get(cantidad)` — retira. **Si no hay suficiente, el proceso se bloquea** hasta que lo haya.
- `yield contenedor.put(cantidad)` — repone. Se bloquea si no cabe.

### Primero, una traza corta

In [ ]:
CONSUMO = 200   # ml de leche por bebida

# Proceso del cliente que llega, espera al barista, es atendido y se va.
def cliente_cafeteria(env, nombre, baristas, leche):
    with baristas.request() as turno:
        yield turno                                   # el cliente espera al barista

        if leche.level < CONSUMO:                     # este if es únicamente para mostrar el mensaje, no es necesario para la simulación
            log(env, f"{nombre}: el barista está libre pero NO HAY LECHE ({leche.level:.0f} ml). Se bloquea.")

        yield leche.get(CONSUMO)                      # aquí intenta retirar leche, y se bloquea si no alcanza
        log(env, f"{nombre}: toma {CONSUMO} ml (quedan {leche.level:.0f} ml) y empieza la bebida")

        yield env.timeout(random.expovariate(2.0))    # prepara la bebida
        log(env, f"{nombre}: bebida lista, se va")

# Proceso que genera clientes y los envía a la cola del barista
def llegadas_cafeteria(env, baristas, leche, lam, n_max):
    for i in range(1, n_max + 1):
        yield env.timeout(random.expovariate(lam))
        env.process(cliente_cafeteria(env, f"Cliente {i}", baristas, leche))  # registramos el proceso del cliente que llega

# Proceso que repone leche cada R minutos
def reposicion(env, leche, lote, R):
    while True:
        yield env.timeout(R)                          # el proveedor llega cada R minutos
        yield leche.put(lote)
        log(env, f">>> LLEGA REPOSICIÓN de {lote} ml (nevera queda en {leche.level:.0f} ml)")


# Simulamos la cafetería con un solo barista, llegadas ~ Exp(0.9) hasta 8 clientes, y reposición de leche cada 8 minutos con lotes de 1500 ml
random.seed(3)
env = simpy.Environment()
baristas = simpy.Resource(env, capacity=1)
leche = simpy.Container(env, init=600, capacity=3000)   # arrancamos con leche para solo 3 bebidas

env.process(llegadas_cafeteria(env, baristas, leche, lam=0.9, n_max=8))     # registramos el proceso de llegadas
env.process(reposicion(env, leche, lote=1500, R=8))                         # registramos el proceso de reposición
env.run(until=25)

t =   0.30 | Cliente 1: toma 200 ml (quedan 400 ml) y empieza la bebida
t =   0.53 | Cliente 1: bebida lista, se va
t =   1.18 | Cliente 2: toma 200 ml (quedan 200 ml) y empieza la bebida
t =   1.67 | Cliente 2: bebida lista, se va
t =   2.20 | Cliente 3: toma 200 ml (quedan 0 ml) y empieza la bebida
t =   2.21 | Cliente 3: bebida lista, se va
t =   2.28 | Cliente 4: el barista está libre pero NO HAY LECHE (0 ml). Se bloquea.
t =   8.00 | >>> LLEGA REPOSICIÓN de 1500 ml (nevera queda en 1300 ml)
t =   8.00 | Cliente 4: toma 200 ml (quedan 1300 ml) y empieza la bebida
t =   8.32 | Cliente 4: bebida lista, se va
t =   8.32 | Cliente 5: toma 200 ml (quedan 1100 ml) y empieza la bebida
t =   9.22 | Cliente 5: bebida lista, se va
t =   9.22 | Cliente 6: toma 200 ml (quedan 900 ml) y empieza la bebida
t =   9.55 | Cliente 6: bebida lista, se va
t =   9.55 | Cliente 7: toma 200 ml (quedan 700 ml) y empieza la bebida
t =  10.06 | Cliente 7: bebida lista, se va
t =  10.97 | Cliente 8: toma 200 

Lo que acaba de pasar:

- Los clientes 1, 2 y 3 se atienden sin problema y **vacían la nevera**.
- El cliente 4 llega en $t \approx 2.3$, agarra al barista, y se queda bloqueado **casi 6 minutos** esperando el camión de la leche. El barista está libre pero no puede hacer nada.
- Los clientes 5, 6 y 7 llegan en ese intervalo y **no aparecen en la traza** hasta después de la reposición: están atrapados en la cola detrás del cliente 4.
- Apenas llega el lote en $t = 8$, todos salen en cascada.

> El mensaje de reposición reporta el nivel **después** de que SimPy ya le entregó la leche al cliente que estaba bloqueado: por eso dice 1300 y no 1500. Los `get` pendientes se atienden en el mismo instante en que entra el `put`.

### Ahora la pregunta interesante

La cafetería quiere reducir el tiempo que los clientes pasan en el local. Hay dos familias de decisiones sobre la mesa:

- **Barista**: contratar uno más, o darle una máquina que lo haga el doble de rápido.
- **Leche**: pedir lotes más grandes, o pedir más seguido.

¿Cuál conviene? Simulemos las dos. Ahora sí con un horizonte largo y midiendo el tiempo promedio en el sistema $W$.

In [ ]:
# Definimos una función que simula la cafetería y devuelve un diccionario con los indicadores agregados. 
# En este caso no usamos prints, ya que el horizonte es largo y queremos obtener los indicadores agregados.
def simular_cafeteria(lam=0.9, mu=2.0, s=1, consumo=200,
                      lote=1500, R=8, nevera=3000, T=5000, semilla=0):

    # Semilla fija para resultados reproducibles
    random.seed(semilla)

    # Diccionario para almacenar los indicadores
    m = {"salidas": 0,          # número de clientes atendidos
         "W": [],               # tiempos en el sistema de cada cliente
         "t_sirviendo": 0.0,    # tiempo total que los baristas estuvieron preparando bebidas
         "t_sin_leche": 0.0}    # tiempo total que los baristas estuvieron bloqueados esperando leche

    # Proceso del cliente que llega, espera al barista, es atendido y se va.
    def cliente(env, baristas, leche):
        t_llegada = env.now
        with baristas.request() as turno:
            yield turno
            t0 = env.now
            yield leche.get(consumo)                  # puede bloquearse
            m["t_sin_leche"] += env.now - t0          # tiempo de barista desperdiciado

            t_serv = random.expovariate(mu)
            m["t_sirviendo"] += t_serv
            yield env.timeout(t_serv)
        m["W"].append(env.now - t_llegada)
        m["salidas"] += 1

    # Proceso que genera clientes y los envía a la cola del barista
    def llegadas(env, baristas, leche):
        while True:
            yield env.timeout(random.expovariate(lam))
            env.process(cliente(env, baristas, leche))

    # Proceso que repone leche cada R minutos
    def reposicion(env, leche):
        while True:
            yield env.timeout(R)
            yield leche.put(lote)

    # Corremos la simulación
    env = simpy.Environment()
    baristas = simpy.Resource(env, capacity=s)
    leche = simpy.Container(env, init=nevera, capacity=nevera)
    env.process(llegadas(env, baristas, leche))
    env.process(reposicion(env, leche))
    env.run(until=T)

    # Devolvemos un diccionario con los indicadores agregados
    return {
        "W": statistics.mean(m["W"]),                 # tiempo promedio en el sistema (min)
        "throughput": m["salidas"] / T,               # clientes atendidos por minuto
        "frac_sirviendo": m["t_sirviendo"] / (T * s), # % del tiempo preparando bebidas
        "frac_sin_leche": m["t_sin_leche"] / (T * s), # % del tiempo bloqueado sin leche
    }

Una sola corrida de una simulación estocástica es **una sola observación**: si cambias la semilla, el resultado cambia. Para que la comparación entre escenarios no dependa de la suerte, vamos a promediar 10 corridas con semillas distintas.

(El tratamiento formal de cuántas réplicas necesito y qué tan precisa es mi estimación es el tema de la semana 14. Por ahora quédate con la idea de que **una corrida no basta**.)

In [18]:
def promedio_replicas(n_replicas=10, **parametros):
    corridas = [simular_cafeteria(semilla=i, **parametros) for i in range(1, n_replicas + 1)]
    return {k: statistics.mean(c[k] for c in corridas) for k in corridas[0]}

escenarios = {
    "Base":                     dict(),
    "A) barista 2x más rápido": dict(mu=4.0),
    "B) dos baristas":          dict(s=2),
    "C) reponer cada 4 min":    dict(R=4),
    "D) lote de 2000 ml":       dict(lote=2000, nevera=4000),
}

filas = []
for nombre, cambios in escenarios.items():
    r = promedio_replicas(**cambios)
    r["escenario"] = nombre
    filas.append(r)

df = pd.DataFrame(filas)[["escenario", "W", "throughput", "frac_sirviendo", "frac_sin_leche"]]

W_base = df.loc[df["escenario"] == "Base", "W"].iloc[0]
df["mejora_en_W_%"] = 100 * (W_base - df["W"]) / W_base

df.round(3)

,escenario,W,throughput,frac_sirviendo,frac_sin_leche,mejora_en_W_%
0,Base,5.564,0.904,0.450,0.216,0.000
1,A) barista 2x más rápido,5.369,0.902,0.224,0.297,3.495
2,B) dos baristas,6.330,0.903,0.225,0.296,-13.768
3,C) reponer cada 4 min,0.913,0.906,0.453,0.000,83.595
4,D) lote de 2000 ml,0.913,0.906,0.453,0.000,83.595


### Discusión

Mira la columna `mejora_en_W_%`:

- **Duplicar la velocidad del barista (A) no sirve de nada.** La mejora es de un par de puntos porcentuales, dentro del ruido de la simulación.
- **Poner un segundo barista (B) tampoco.** De hecho sale peor. Con dos baristas hay **dos** clientes bloqueados esperando leche al mismo tiempo, así que se desperdicia el doble de capacidad.
- **Modificar la reposición o el lote de la leche (C o D) reduce el tiempo en el sistema en más del 80%**, y hace desaparecer por completo el tiempo bloqueado.

La explicación está en las dos últimas columnas. En el escenario base el barista pasa **~45% del tiempo preparando bebidas y ~20% del tiempo bloqueado sin leche**. Es decir: *ya estaba desocupado más de la mitad del tiempo*. Darle más velocidad o un compañero es agregar capacidad donde no falta. Peor aún: en el escenario A el barista sirve más rápido, así que llega antes al `get` de la leche y su tiempo bloqueado **sube** del 20% al 30%.

> **La moraleja.** La capacidad efectiva del sistema no la pone el recurso más visible, sino el **más restrictivo**. Aquí la leche entrega $1500 / (8 \times 200) = 0.94$ bebidas por minuto, mientras que llegan $\lambda = 0.9$ clientes por minuto: el margen es de apenas 4%. El barista, en cambio, podría con 2.0 bebidas por minuto. **El cuello de botella es el inventario, no el servidor.**

Esto es difícil de ver sin simular, y es una de las razones por las que la simulación vale la pena: el modelo analítico de una M/M/1 le habría dicho a la cafetería que contratara otro barista.

**Para discutir en clase:**

1. ¿Qué pasaría si el `yield leche.get(...)` estuviera **antes** del `with baristas.request()`? ¿Cambiaría la conclusión? ¿Cuál de los dos modelos describe mejor una cafetería real?
2. El escenario D (lote más grande) y el C (reponer más seguido) dan prácticamente el mismo resultado. ¿Por qué? ¿En qué se diferencian en costo para la cafetería?
3. ¿Qué tendría que pasar para que el barista **sí** volviera a ser el cuello de botella?


<details>
<summary><b>Respuestas:</b> (ábrela solo después de hacer tu propio análisis)</summary>

<br>

**¿De verdad las pensaste? 0_0**

Sabemos que es fácil rendirse a la primera, pero queremos que realmente lo intentes.

Escribe tus respuestas en la celda de abajo, discútelas con quien tengas al lado, y después contrástalas con tu asistente graduado.

</details>

## 6. Ejercicio guiado: el lavadero de carros

Un lavadero tiene **2 bahías** de lavado. Los carros llegan según un proceso de Poisson con tasa $\lambda = 0.25$ carros/min y el lavado dura un tiempo exponencial de media 6 min. Cada lavado consume **80 litros** de agua de un tanque de **2000 litros**, que se rellena hasta el tope cada **90 minutos**.

La celda de abajo **ya corre**, pero el modelo está incompleto: los carros no piden bahía ni consumen agua, así que salen limpios al instante y nunca hay cola. Reemplaza cada `# TODO` y borra los `placeholder` para que el modelo quede bien.

In [ ]:
LAM_LAVADERO = 0.25        # carros por minuto
MEDIA_LAVADO = 6.0         # minutos
AGUA_POR_LAVADO = 80       # litros
TANQUE = 2000              # litros
CADA_CUANTO_RELLENAN = 90  # minutos

def carro(env, nombre, bahias, agua):
    log(env, f"{nombre} llega")

    # TODO 1: pedir una bahía  ->  with bahias.request() as turno: yield turno
    #         (todo lo que sigue debe quedar DENTRO del bloque 'with')
    # TODO 2: retirar AGUA_POR_LAVADO litros del tanque 'agua'
    # TODO 3: el lavado tarda Exp con media MEDIA_LAVADO
    #         Ojo: random.expovariate recibe la TASA, no la media.

    yield env.timeout(0)   # <-- placeholder: bórralo al completar los TODO

    log(env, f"{nombre} sale limpio")


def llegadas_lavadero(env, bahias, agua, n_max):
    for i in range(1, n_max + 1):
        # TODO 4: reemplaza el interarribo fijo por uno Exp(LAM_LAVADERO)
        yield env.timeout(4.0)   # <-- placeholder
        env.process(carro(env, f"Carro {i}", bahias, agua))


def rellenar_tanque(env, agua):
    while True:
        yield env.timeout(CADA_CUANTO_RELLENAN)
        faltante = agua.capacity - agua.level         # lo llenamos hasta el tope
        if faltante > 0:
            yield agua.put(faltante)
            log(env, f">>> Se rellena el tanque (queda en {agua.level:.0f} L)")


random.seed(1)
env = simpy.Environment()
bahias = simpy.Resource(env, capacity=2)
agua = simpy.Container(env, init=TANQUE, capacity=TANQUE)

env.process(llegadas_lavadero(env, bahias, agua, n_max=12))
env.process(rellenar_tanque(env, agua))
env.run(until=60)

**Preguntas sobre tu modelo (una vez completado):**

1. ¿El tanque alcanza a quedarse vacío en 200 minutos? Compara la tasa de consumo ($\lambda \times 80$ L/min) contra la de reposición ($2000/90$ L/min). ¿Cuál manda?
2. Cambia `capacity=2` a `capacity=1`. ¿Qué le pasa a la traza?
3. ¿Cuál es el cuello de botella: las bahías o el agua? Sube `CADA_CUANTO_RELLENAN` a 300 y vuelve a mirar.

<details>
<summary><b>Solución</b> (ábrela solo después de intentarlo)</summary>

<br>

**Casi ;)** 

Sabemos que abriste esto de una. No pasa nada, pero de verdad inténtalo primero. Son cuatro líneas y ya las viste en la cafetería.

Pista: todo lo que necesitas está en la sección 5. El `with ... request()`, el `get` del contenedor y el `expovariate` ya los usaste hoy. Lo único nuevo es que `random.expovariate` recibe la **tasa**, y a ti te dieron la **media**.

Cuando tengas tu versión corriendo, compárala con tus compañeros y muéstrasela a tu asistente graduado para revisar.

## Resumen

Todo lo de hoy cabe en seis líneas:

```python
env = simpy.Environment()                       # el reloj y el planificador
env.process(mi_proceso(env))                    # registrar una entidad
yield env.timeout(t)                            # dejar pasar t unidades de tiempo
yield env.timeout(random.expovariate(lam))      # ... un tiempo aleatorio Exp(lam)
with recurso.request() as turno: yield turno    # pedir un servidor y hacer cola
yield contenedor.get(cantidad)                  # consumir de un inventario
```

Las tres ideas principales:

1. **El reloj salta entre eventos.** `env.now` solo toma los valores en que ocurre algo. Sin `yield` no hay avance del tiempo.
2. **Un proceso es una función con `yield`.** Se escribe una vez y se instancia tantas veces como entidades haya.
3. **Un `yield` puede bloquear indefinidamente.** No solo `timeout` hace esperar: pedir un recurso ocupado o sacar de un contenedor vacío también. Ahí es donde aparecen las colas y los cuellos de botella.